# Phase 6 — Clinical Decision Support System Development

## Step 1 — Define the CDSS Design

### Inputs
The CDSS will use:
- Patient demographics
- Symptoms/chief complaint
- Vital signs
- Medical history
- Allergies
- Current medications, if available
- Laboratory results, if available
- Phase 5 ESI/triage result
- Patient narrative

### Outputs
The CDSS should provide structured decision support such as:
```json
{
  "risk_level": "High",
  "possible_conditions": [],
  "recommended_tests": [],
  "treatment_considerations": [],
  "clinical_reasoning": "",
  "warnings": [],
  "requires_human_review": true
}

In [1]:
# STEP 1: ENVIRONMENT & DIRECTORY SETUP

import sys
import os
import gc
import torch
import warnings
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Clear VRAM & Memory Garbage
gc.collect()
torch.cuda.empty_cache()
warnings.filterwarnings("ignore", category=UserWarning)

# 3. Configure Phase 6 Paths
phase6_dir = "/content/drive/MyDrive/MIMIC-IV-Ext Triage Instruction Corpus/Final_Files/Phase_6_Final"
files_dir = os.path.join(phase6_dir, "Files")

# Ensure directories exist
os.makedirs(files_dir, exist_ok=True)

for path in [files_dir, phase6_dir]:
    if path in sys.path:
        sys.path.remove(path)
    sys.path.insert(0, path)

os.chdir(phase6_dir)
print(f"Environment ready. Current Directory: {os.getcwd()}")

Mounted at /content/drive
Environment ready. Current Directory: /content/drive/MyDrive/MIMIC-IV-Ext Triage Instruction Corpus/Final_Files/Phase_6_Final


#Patient Data & Clinical Input Processing

## Overview & Goal
This part is about the entry point of the CDSS pipeline. Their goal is to retrieve raw patient records, extract clinical features (demographics, vital signs, chief complaints/narratives), validate required data fields, integrate the Phase 5 ESI triage results, and structure everything into a standardized JSON format expected by downstream systems.

---

## Technical Responsibilities

1. **Database & API Connectivity**: Retrieve raw patient files.
2. **Clinical Validation**: Ensure critical fields (e.g., age, vital signs, narrative text) exist and fall within physically plausible ranges (e.g., HR between 20–250 bpm).
3. **Data Standardization**: Convert raw text and structured inputs into a unified dictionary structure.
4. **Phase 5 Triage Integration**: Merge the ML/XGBoost ESI triage level (and confidence score) into the primary clinical payload.

---

## Data Pipeline Architecture

1. Raw Patient Database / Record

2. Field Extraction & Cleaning

3. Clinical Data Validation
(Range checks & missing fields)

4. Phase 5 ESI Triage Binding

5. Standardized Clinical Payload (JSON)

---

## Output Spec (`clinical_input.py`)

The processed payload generated by this part follows this schema:

```json
{
  "patient_id": 101,
  "patient": {
    "age": 65,
    "gender": "Male"
  },
  "vitals": {
    "heart_rate": 108,
    "sbp": 155,
    "dbp": 92,
    "spo2": 94,
    "temperature": 37.1,
    "respiratory_rate": 22
  },
  "symptoms": "Severe chest pain radiating to left arm, shortness of breath, diaphoresis",
  "medical_history": ["Hypertension", "Hyperlipidemia"],
  "allergies": ["Penicillin"],
  "triage": {
    "esi": 2,
    "confidence": 0.92,
    "source": "Phase 5 XGBoost Triage Model"
  }
  }

In [2]:
# WRITE clinical_input.py TO DRIVE
code_clinical_input = """'''
clinical_input.py
Patient Data & Clinical Input Processing
'''

import os
import json
import pandas as pd

try:
    from triage_module import predict_triage, FEATURE_COLUMNS
except ModuleNotFoundError:
    from Files.triage_module import predict_triage, FEATURE_COLUMNS


def get_clinical_input(patient_id: int, csv_path: str = None) -> dict:
    \"\"\"
    Retrieves patient data, handles missing fields, fetches the triage score,
    and standardizes the output for the CDSS.
    \"\"\"
    if csv_path is None:
        possible_paths = [
            "Files/preprocessed_dataset.csv",
            "preprocessed_dataset.csv",
            "../Files/preprocessed_dataset.csv",
            "/content/drive/MyDrive/MIMIC-IV-Ext Triage Instruction Corpus/Final_Files/Phase_6_Final/Files/preprocessed_dataset.csv",
        ]
        for path in possible_paths:
            if os.path.exists(path):
                csv_path = path
                break

    if not csv_path or not os.path.exists(csv_path):
        return {"error": "Database file 'preprocessed_dataset.csv' not found."}

    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        return {"error": f"Failed to read database: {str(e)}"}

    if patient_id < 0 or patient_id >= len(df):
        return {"error": f"Patient ID {patient_id} not found. Range: 0 to {len(df)-1}"}

    row = df.iloc[patient_id]

    def safe_get(val, default="Not available"):
        return val if pd.notnull(val) else default

    narrative_text = str(row.get("triagecase", row.get("chiefcomplaint", "No narrative available.")))

    structured_feats = {col: row[col] for col in FEATURE_COLUMNS if col in row}
    triage_result = predict_triage(structured_feats, narrative_text)

    cdss_input = {
        "patient_id": int(patient_id),
        "patient": {
            "age": safe_get(row.get("age")),
            "gender": "Male" if row.get("gender_M") == 1.0 else "Female",
        },
        "symptoms": narrative_text,
        "vitals": {
            "heart_rate": safe_get(row.get("heartrate")),
            "sbp": safe_get(row.get("sbp")),
            "dbp": safe_get(row.get("dbp")),
            "spo2": safe_get(row.get("o2sat")),
            "temperature": safe_get(row.get("temperature")),
        },
        "medical_history": [],
        "allergies": [],
        "medications": [],
        "laboratory_results": [],
        "narrative": narrative_text,
        "triage": {
            "esi": triage_result.get("predicted_esi", 0),
            "confidence": triage_result.get("confidence", 0.0),
        },
    }

    return cdss_input
"""

files_dir = "/content/drive/MyDrive/MIMIC-IV-Ext Triage Instruction Corpus/Final_Files/Phase_6_Final/Files"
os.makedirs(files_dir, exist_ok=True)

with open(os.path.join(files_dir, "clinical_input.py"), "w") as f:
    f.write(code_clinical_input)

print("clinical_input.py saved successfully to Files directory")

clinical_input.py saved successfully to Files directory


In [3]:
# WRITE clinical_input_api.py TO DRIVE
code_clinical_api = """'''
clinical_input_api.py
FastAPI Interface for Clinical Data Retrieval
'''

from fastapi import APIRouter, HTTPException
from clinical_input import get_clinical_input

router = APIRouter()

@router.get("/api/clinical-input/{patient_id}")
def fetch_patient_data(patient_id: int):
    \"\"\"
    API endpoint to retrieve the standardized CDSS input for a specific patient.
    \"\"\"
    data = get_clinical_input(patient_id)

    if "error" in data:
        raise HTTPException(status_code=404, detail=data["error"])

    return data
"""

with open(os.path.join(files_dir, "clinical_input_api.py"), "w") as f:
    f.write(code_clinical_api)

print("clinical_input_api.py saved successfully to Files directory")

clinical_input_api.py saved successfully to Files directory


In [5]:
import os
import json

phase6_dir = "/content/drive/MyDrive/MIMIC-IV-Ext Triage Instruction Corpus/Final_Files/Phase_6_Final"
files_dir = os.path.join(phase6_dir, "Files")

# 1. Path to triage_module.py
triage_module_path = os.path.join(files_dir, "triage_module.py")

# 2. Rewrite triage_module.py with dynamic model loading
triage_module_code = '''"""
triage_module.py
Machine Learning Triage Prediction Module (Phase 5 Model Loader)
"""

import os
import joblib
import pandas as pd

FEATURE_COLUMNS = [
    "age", "heartrate", "sbp", "dbp", "o2sat", "temperature", "respiratory_rate"
]

def _find_model_file():
    possible_paths = [
        "triage_ml_model.pkl",
        "Files/triage_ml_model.pkl",
        "/content/drive/MyDrive/MIMIC-IV-Ext Triage Instruction Corpus/Final_Files/Phase_6_Final/Files/triage_ml_model.pkl",
        "/content/drive/MyDrive/MIMIC-IV-Ext Triage Instruction Corpus/Final_Files/Phase_6/Files/triage_ml_model.pkl",
    ]
    for path in possible_paths:
        if os.path.exists(path):
            return path
    raise FileNotFoundError("Could not locate 'triage_ml_model.pkl' in drive directory.")

_model_cache = None

def load_triage_model():
    global _model_cache
    if _model_cache is None:
        model_path = _find_model_file()
        _model_cache = joblib.load(model_path)
    return _model_cache

def predict_triage(structured_feats: dict, narrative_text: str = "") -> dict:
    """
    Predicts ESI level and confidence score using trained model features.
    """
    try:
        model = load_triage_model()

        # Build single-row DataFrame matching feature order
        input_data = {}
        for col in FEATURE_COLUMNS:
            val = structured_feats.get(col, None)
            if val is None or pd.isna(val) or val == "Not available":
                input_data[col] = [0.0]
            else:
                input_data[col] = [float(val)]

        df_input = pd.DataFrame(input_data)

        # Predict ESI
        pred = model.predict(df_input)[0]

        # Get probability/confidence if model supports it
        confidence = 0.85
        if hasattr(model, "predict_proba"):
            probs = model.predict_proba(df_input)[0]
            confidence = float(max(probs))

        return {
            "predicted_esi": int(pred),
            "confidence": round(confidence, 4)
        }
    except Exception as e:
        # Fallback response if model fails
        return {
            "predicted_esi": 3,
            "confidence": 0.50,
            "error": str(e)
        }
'''

with open(triage_module_path, "w") as f:
    f.write(triage_module_code)

print("Updated triage_module.py with dynamic path loading")

# 3. Import sys and refresh Python module cache
import sys
for p in [files_dir, phase6_dir]:
    if p in sys.path:
        sys.path.remove(p)
    sys.path.insert(0, p)

os.chdir(phase6_dir)

# 4. Test the pipeline again
from clinical_input import get_clinical_input

patient_data = get_clinical_input(0)
print("\nCLINICAL INPUT OUTPUT FOR PATIENT 0")
print(json.dumps(patient_data, indent=2))

Updated triage_module.py with dynamic path loading

CLINICAL INPUT OUTPUT FOR PATIENT 0
{
  "patient_id": 0,
  "patient": {
    "age": 1.0045660403630117,
    "gender": "Male"
  },
  "symptoms": "s/p Fall, Head injury",
  "vitals": {
    "heart_rate": -1.0521477438221538,
    "sbp": 1.264178231175774,
    "dbp": -0.4464988791843108,
    "spo2": -0.0840387788370226,
    "temperature": 0.0145171676284753
  },
  "medical_history": [],
  "allergies": [],
  "medications": [],
  "laboratory_results": [],
  "narrative": "s/p Fall, Head injury",
  "triage": {
    "esi": 3,
    "confidence": 0.5
  }
}


# Clinical Risk & Recommendation Engine

## Overview & Goal
This module develops the core deterministic clinical decision-support logic. It takes the standardized patient record and Phase 5 ESI triage score generated in the previous step and evaluates them against clinical rules to produce structured risk assessments, potential conditions to consider, recommended diagnostic tests, treatment considerations, emergency warnings, and flags for human clinician review.

---

## Technical Responsibilities

1. **Vital Sign Threshold Analysis**: Evaluate heart rate, blood pressure, oxygen saturation, and temperature against standard clinical thresholds (e.g., severe hypotension, tachycardia, hypoxia).
2. **Rule-Based Condition Mapping**: Cross-reference primary symptoms/chief complaints (e.g., chest pain, shortness of breath, sudden weakness) with clinical rule sets.
3. **Risk Tier Assignment**: Assign an overall risk level (`High`, `Moderate`, or `Low`) based on vital sign instability and triage level.
4. **Actionable Recommendations**: Output non-prescriptive recommendations for diagnostic evaluations (e.g., 12-lead ECG, cardiac biomarkers, chest X-ray) and immediate interventions according to emergency protocols.

---

## Module Workflow

1. Standardized Clinical Payload + ESI Triage Score

2. Vital Signs Threshold Check

3. Risk Level Determination
(High / Moderate / Low)

4. Clinical Rule Match Engine
(Symptom + Vitals -> Potential Concerns)

5. Structured Recommendations & Safety Warnings

---

## Expected Output Schema

```json
{
  "risk_level": "High",
  "possible_conditions": [
    "Acute coronary syndrome should be considered."
  ],
  "recommended_tests": [
    "12-lead ECG",
    "Troponin / Cardiac Biomarkers",
    "Routine blood work (CBC, BMP)"
  ],
  "treatment_considerations": [
    "Continuous cardiac monitoring",
    "Supplemental oxygen if SpO2 < 94%",
    "Immediate clinician evaluation"
  ],
  "warnings": [
    "CRITICAL: Hemodynamic instability or high-risk presentation detected."
  ],
  "requires_human_review": true
}

In [6]:
import os
import sys
import json

# Define directories
phase6_dir = "/content/drive/MyDrive/MIMIC-IV-Ext Triage Instruction Corpus/Final_Files/Phase_6_Final"
files_dir = os.path.join(phase6_dir, "Files")
os.makedirs(files_dir, exist_ok=True)

# 1. WRITE risk_assessment.py
code_risk_assessment = '''"""
risk_assessment.py
Clinical Risk Assessment Engine
Rule-based risk stratification that combines the Phase 5 ESI triage level
with vital-sign red flags to produce a decision-support risk assessment.
"""

from typing import Dict, Any, List

def _to_float(value, default=None):
    """Safely convert a vitals value to float."""
    try:
        return float(value)
    except (TypeError, ValueError):
        return default

def _check_vital_red_flags(vitals: Dict[str, Any]) -> List[str]:
    """Inspect vital signs for values outside typical safe ranges."""
    flags = []

    hr = _to_float(vitals.get("heart_rate"))
    sbp = _to_float(vitals.get("sbp"))
    dbp = _to_float(vitals.get("dbp"))
    spo2 = _to_float(vitals.get("spo2"))
    temp = _to_float(vitals.get("temperature"))

    if hr is not None:
        if hr >= 130:
            flags.append("Severe tachycardia detected (HR >= 130 bpm).")
        elif hr >= 100:
            flags.append("Tachycardia detected (HR >= 100 bpm).")
        elif hr <= 50:
            flags.append("Bradycardia detected (HR <= 50 bpm).")

    if sbp is not None:
        if sbp < 90:
            flags.append("Severe hypotension detected (SBP < 90 mmHg).")
        elif sbp >= 180:
            flags.append("Hypertensive urgency range (SBP >= 180 mmHg).")

    if dbp is not None and dbp >= 120:
        flags.append("Severely elevated diastolic pressure (DBP >= 120 mmHg).")

    if spo2 is not None:
        if spo2 < 90:
            flags.append("Severe hypoxia detected (SpO2 < 90%).")
        elif spo2 < 94:
            flags.append("Mild-to-moderate hypoxia detected (SpO2 < 94%).")

    if temp is not None:
        if temp >= 39.0:
            flags.append("High fever detected (Temp >= 39.0C).")
        elif temp <= 35.0:
            flags.append("Hypothermia detected (Temp <= 35.0C).")

    # Shock index (HR / SBP) check
    if hr is not None and sbp not in (None, 0):
        shock_index = hr / sbp
        if shock_index >= 1.0:
            flags.append(
                f"Elevated shock index ({shock_index:.2f}) suggests possible hemodynamic compromise."
            )

    return flags

def _esi_to_baseline_risk(esi) -> str:
    """Map Phase 5 ESI level to a baseline risk category."""
    try:
        esi = int(esi)
    except (TypeError, ValueError):
        return "Unknown"

    if esi in (1, 2):
        return "High"
    if esi == 3:
        return "Moderate"
    if esi in (4, 5):
        return "Low"
    return "Unknown"

_RISK_ORDER = {"Low": 0, "Moderate": 1, "High": 2, "Unknown": 1}

def _escalate(risk_a: str, risk_b: str) -> str:
    """Return whichever of the two risk levels is more severe."""
    return risk_a if _RISK_ORDER.get(risk_a, 1) >= _RISK_ORDER.get(risk_b, 1) else risk_b

def assess_risk(clinical_input: Dict[str, Any]) -> Dict[str, Any]:
    """Main entry point for risk assessment."""
    vitals = clinical_input.get("vitals", {}) or {}
    triage = clinical_input.get("triage", {}) or {}
    esi = triage.get("esi")

    baseline_risk = _esi_to_baseline_risk(esi)
    red_flags = _check_vital_red_flags(vitals)

    severe_keywords = ("Severe", "shock index")
    vital_escalates_to_high = any(
        any(keyword in flag for keyword in severe_keywords) for flag in red_flags
    )

    risk_level = baseline_risk
    if vital_escalates_to_high:
        risk_level = _escalate(risk_level, "High")
    elif red_flags:
        risk_level = _escalate(risk_level, "Moderate")

    warnings = list(red_flags)
    if risk_level == "High":
        warnings.append("Immediate clinician assessment required.")
    if esi in (None, "Unknown") or baseline_risk == "Unknown":
        warnings.append("Triage (ESI) result missing or invalid; risk level may be incomplete.")

    requires_human_review = risk_level in ("High", "Unknown") or bool(red_flags)

    return {
        "risk_level": risk_level,
        "risk_factors": red_flags,
        "warnings": warnings,
        "requires_human_review": requires_human_review,
    }
'''

with open(os.path.join(files_dir, "risk_assessment.py"), "w") as f:
    f.write(code_risk_assessment)

print("Successfully saved risk_assessment.py")

# 2. WRITE recommendation_engine.py
code_recommendation_engine = '''"""
recommendation_engine.py
Clinical Recommendation Engine
Rule-based generation of possible conditions, recommended diagnostic tests,
and treatment considerations, built on top of risk_assessment.py.
"""

from typing import Dict, Any, List

try:
    from risk_assessment import assess_risk
except ModuleNotFoundError:
    from Files.risk_assessment import assess_risk

_SYMPTOM_RULES = [
    {
        "keywords": ["chest pain", "chest tightness", "radiating to left arm", "pressure in chest"],
        "possible_conditions": ["Acute coronary syndrome should be considered"],
        "recommended_tests": ["ECG", "Cardiac biomarkers (troponin)"],
        "treatment_considerations": ["Continuous cardiac monitoring"],
    },
    {
        "keywords": ["shortness of breath", "difficulty breathing", "dyspnea", "can't breathe"],
        "possible_conditions": [
            "Acute respiratory distress or pulmonary etiology should be considered"
        ],
        "recommended_tests": ["Chest X-ray", "Pulse oximetry trend", "Arterial blood gas"],
        "treatment_considerations": ["Supplemental oxygen as clinically indicated"],
    },
    {
        "keywords": ["fever", "chills", "sepsis", "infection"],
        "possible_conditions": ["Infectious or septic process should be considered"],
        "recommended_tests": ["Complete blood count", "Blood cultures", "Lactate level"],
        "treatment_considerations": ["Consider sepsis protocol per institutional guidelines"],
    },
    {
        "keywords": ["severe headache", "worst headache", "sudden headache"],
        "possible_conditions": [
            "Intracranial hemorrhage or neurological emergency should be considered"
        ],
        "recommended_tests": ["CT head (non-contrast)", "Neurological exam"],
        "treatment_considerations": ["Urgent neurology consultation may be warranted"],
    },
    {
        "keywords": ["abdominal pain", "stomach pain"],
        "possible_conditions": ["Acute abdominal process should be considered"],
        "recommended_tests": ["Abdominal exam", "Basic metabolic panel", "Lipase"],
        "treatment_considerations": ["Surgical consultation if peritoneal signs present"],
    },
]

def _match_symptom_rules(narrative_text: str) -> Dict[str, List[str]]:
    """Scan narrative text for known keyword patterns and aggregate matched suggestions."""
    text = (narrative_text or "").lower()

    possible_conditions: List[str] = []
    recommended_tests: List[str] = []
    treatment_considerations: List[str] = []

    for rule in _SYMPTOM_RULES:
        if any(keyword in text for keyword in rule["keywords"]):
            possible_conditions.extend(rule["possible_conditions"])
            recommended_tests.extend(rule["recommended_tests"])
            treatment_considerations.extend(rule["treatment_considerations"])

    def dedupe(items):
        seen = set()
        out = []
        for item in items:
            if item not in seen:
                seen.add(item)
                out.append(item)
        return out

    return {
        "possible_conditions": dedupe(possible_conditions),
        "recommended_tests": dedupe(recommended_tests),
        "treatment_considerations": dedupe(treatment_considerations),
    }

def generate_recommendations(clinical_input: Dict[str, Any]) -> Dict[str, Any]:
    """Combines risk assessment with symptom-based recommendations."""
    risk = assess_risk(clinical_input)

    narrative = clinical_input.get("narrative") or clinical_input.get("symptoms") or ""
    matched = _match_symptom_rules(narrative)

    possible_conditions = matched["possible_conditions"]
    recommended_tests = matched["recommended_tests"]
    treatment_considerations = matched["treatment_considerations"]

    if not possible_conditions:
        possible_conditions = [
            "No specific rule-based pattern matched; clinician review recommended."
        ]

    if risk["risk_level"] == "High" and "Immediate clinician assessment" not in treatment_considerations:
        treatment_considerations.append("Immediate clinician assessment")

    return {
        "risk_level": risk["risk_level"],
        "possible_conditions": possible_conditions,
        "recommended_tests": recommended_tests,
        "treatment_considerations": treatment_considerations,
        "warnings": risk["warnings"],
        "requires_human_review": risk["requires_human_review"],
    }
'''

with open(os.path.join(files_dir, "recommendation_engine.py"), "w") as f:
    f.write(code_recommendation_engine)

print("Successfully saved recommendation_engine.py")

# 3. TEST MODULE IN COLAB WITH PATIENT 0
for p in [files_dir, phase6_dir]:
    if p in sys.path:
        sys.path.remove(p)
    sys.path.insert(0, p)

os.chdir(phase6_dir)

from clinical_input import get_clinical_input
from recommendation_engine import generate_recommendations

patient_input = get_clinical_input(0)
recommendations = generate_recommendations(patient_input)

print("\nCLINICAL RECOMMENDATIONS OUTPUT FOR PATIENT 0")
print(json.dumps(recommendations, indent=2))

Successfully saved risk_assessment.py
Successfully saved recommendation_engine.py

CLINICAL RECOMMENDATIONS OUTPUT FOR PATIENT 0
{
  "risk_level": "High",
  "possible_conditions": [
    "No specific rule-based pattern matched; clinician review recommended."
  ],
  "recommended_tests": [],
  "treatment_considerations": [
    "Immediate clinician assessment"
  ],
  "warnings": [
    "Bradycardia detected (HR <= 50 bpm).",
    "Severe hypotension detected (SBP < 90 mmHg).",
    "Severe hypoxia detected (SpO2 < 90%).",
    "Hypothermia detected (Temp <= 35.0C).",
    "Immediate clinician assessment required."
  ],
  "requires_human_review": true
}


# LLM-Based Clinical Decision Support Layer

## Goal
Integrate the fine-tuned **Qwen2.5-3B-Instruct** model (from Phase 4) with the ESI triage prediction (from Phase 5) to generate a structured, natural-language clinical rationale alongside decision-support recommendations.

While Person B handles deterministic rule-based algorithms, Person C manages the generative LLM reasoning layer to summarize complex clinical presentations.

---

## Architecture Workflow

1. Patient Information

2. Phase 5 ESI Triage Result

3. Controlled Clinical Prompt (Chat Template)

4. Fine-tuned Qwen Model (4-bit Quantized / PEFT)

5. Clinical Reasoning & Recommendation Generation

6. Structured JSON Parsing & Safety Overrides

7. Unified CDSS Output

---

## Core Responsibilities & Tasks
1. **Model Loading**: Load base `Qwen/Qwen2.5-3B-Instruct` using 4-bit `BitsAndBytesConfig` and attach the fine-tuned PEFT LoRA adapter.
2. **Context Synthesis**: Combine patient demographics, vitals, narrative chief complaint, and Phase 5 ESI score into a controlled prompt.
3. **Generative CDSS**: Prompt the model to output clinical reasoning, differential considerations, diagnostic tests, and urgent warnings in JSON format.
4. **Robust Parsing Engine**: Utilize multi-tiered parsing (strict JSON parsing + universal regex extractor) to handle raw or imperfect LLM outputs gracefully.
5. **Safety Guardrails**: Automatically enforce `requires_human_review = True` and attach risk-discrepancy warnings for high-acuity patients (ESI 1–2).

---

## Deliverables
- `Files/cdss_llm.py`: Main module for loading model, generating text, and parsing structured output.
- `Files/cdss_llm_api.py`: FastAPI application wrapping the LLM reasoning pipeline into RESTful endpoints.

In [7]:
!pip install -q -U bitsandbytes accelerate peft transformers

In [8]:
import os
import sys
import json
import importlib

# 1. Clear cached module from Python's RAM memory
if "cdss_llm" in sys.modules:
    del sys.modules["cdss_llm"]

# Set folder structure
phase6_dir = "/content/drive/MyDrive/MIMIC-IV-Ext Triage Instruction Corpus/Final_Files/Phase_6_Final"
files_dir = os.path.join(phase6_dir, "Files")
os.makedirs(files_dir, exist_ok=True)

# 1. WRITE cdss_llm.py
code_cdss_llm = r'''"""
cdss_llm.py — LLM-based clinical decision support module (Person C, Phase 6).
Exposes generate_cdss_llm(clinical_input: dict) -> dict
"""

import re
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_PATH = "/content/drive/MyDrive/MIMIC-IV-Ext Triage Instruction Corpus/Final_Files/Phase_6_Final/Files/fine_tuned_qwen/fine_tuned_qwen"

VALID_RISK_LEVELS = {"Low", "Moderate", "High", "Critical"}

CDSS_SYSTEM_PROMPT = """You are an AI clinical decision support assistant for emergency department staff.
You do NOT diagnose or prescribe treatment. You provide decision-support recommendations only, using
cautious clinical language such as "should be considered", "may be indicated", "consider evaluating for".

Given a patient's clinical information and their ESI (triage) level, respond with ONLY a JSON object in
this exact schema, no other text, no markdown code fences, no comments:

{
  "risk_level": "<Low, Moderate, High, or Critical>",
  "possible_conditions": ["<condition to consider>", ...],
  "recommended_tests": ["<test to consider>", ...],
  "treatment_considerations": ["<consideration>", ...],
  "clinical_reasoning": "<1-3 sentence explanation grounded only in the provided information>",
  "warnings": ["<any urgent safety warning, if applicable>"],
  "requires_human_review": <true or false>
}

Rules:
- Do not invent clinical details not present in the input.
- ESI 1-2 should generally set requires_human_review to true.
- If information is insufficient or contradictory, say so in clinical_reasoning and set requires_human_review to true.
- Output ONLY the JSON object. No explanation before or after it.
"""

# Model loading

_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

_base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, quantization_config=_bnb_config, device_map="auto"
)
model = PeftModel.from_pretrained(_base_model, ADAPTER_PATH)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)


# Prompt construction

def build_clinical_summary(clinical_input: dict) -> str:
    patient = clinical_input.get("patient", {})
    vitals = clinical_input.get("vitals", {})
    symptoms = clinical_input.get("symptoms", "Not specified")
    history = clinical_input.get("medical_history", [])
    esi = clinical_input.get("triage", {}).get("esi", "Unknown")

    vitals_str = ", ".join(f"{k.replace('_', ' ').upper()}: {v}" for k, v in vitals.items()) or "Not recorded"
    history_str = ", ".join(history) if history else "None reported"

    return f"""Patient: {patient.get('age', 'Unknown')}-year-old {patient.get('gender', 'Unknown')}
Chief complaint / symptoms: {symptoms}
Vitals: {vitals_str}
Medical history: {history_str}
ESI (triage) level: {esi}"""


def build_cdss_prompt(clinical_input: dict):
    summary = build_clinical_summary(clinical_input)
    messages = [
        {"role": "system", "content": CDSS_SYSTEM_PROMPT},
        {"role": "user", "content": summary},
    ]
    return tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)


# Generation

def generate_cdss_response(clinical_input: dict, max_new_tokens: int = 600) -> str:
    inputs = build_cdss_prompt(clinical_input)
    prompt_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated_ids = output[0][prompt_len:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


# Parsing

FIELD_PATTERNS = {
    "risk_level": r'"?risk_level"?\s*:?\s*',
    "possible_conditions": r'(?:\d+\.\s*)?"?possible[_ ]conditions"?\s*:?\s*',
    "recommended_tests": r'(?:\d+\.\s*)?"?recommended[_ ]tests"?\s*:?\s*',
    "treatment_considerations": r'(?:\d+\.\s*)?"?treatment[_ ]considerations"?\s*:?\s*',
    "clinical_reasoning": r'(?:\d+\.\s*)?"?clinical[_ ]reasoning"?\s*:?\s*',
    "warnings": r'(?:\d+\.\s*)?"?warnings"?\s*:?\s*',
    "requires_human_review": r'(?:\d+\.\s*)?"?requires[_ ]human[_ ]review"?\s*:?\s*',
}


def extract_list_field(section_text: str):
    section_text = section_text.strip()

    bracket_match = re.search(r"\[(.*?)\]", section_text, re.DOTALL)
    if bracket_match:
        inner = bracket_match.group(1)
        items = re.findall(r'"([^"]*)"', inner)
        if items:
            return [i.strip() for i in items if i.strip()]
        if inner.strip() == "":
            return []

    if re.search(r'(?:^|\s)-\s+\S', section_text):
        items = re.split(r'\s*-\s+', section_text)
        return [i.strip(' "\'.,') for i in items if i.strip()]

    return []


def extract_scalar_field(section_text: str):
    section_text = section_text.strip()
    quoted = re.search(r'"([^"]*)"', section_text)
    if quoted and len(quoted.group(1)) > 3:
        return quoted.group(1).strip()

    cleaned = re.sub(r'^[:\-\s"\[]+', '', section_text)
    cleaned = re.split(r'\s{2,}|"\s*,', cleaned)[0]
    return cleaned.strip(' "\'.,')


def universal_field_extract(raw_text):
    matches = []
    for field_name, pattern in FIELD_PATTERNS.items():
        for m in re.finditer(pattern, raw_text, re.IGNORECASE):
            matches.append((m.start(), m.end(), field_name))
    if not matches:
        return {}
    matches.sort(key=lambda x: x[0])
    dedup, seen = [], set()
    for start, end, name in matches:
        if name in seen:
            continue
        seen.add(name)
        dedup.append((start, end, name))
    dedup.sort(key=lambda x: x[0])
    sections = {}
    for i, (start, end, name) in enumerate(dedup):
        next_start = dedup[i + 1][0] if i + 1 < len(dedup) else len(raw_text)
        sections[name] = raw_text[end:next_start]
    return sections


def build_result_from_sections(sections):
    risk_level = "Unknown"
    rl_section = sections.get("risk_level", "")
    m = re.search(r"\b(Low|Moderate|High|Critical)\b", rl_section[:60], re.IGNORECASE)
    if m:
        risk_level = m.group(1).capitalize()

    review_text = sections.get("requires_human_review", "")
    requires_human_review = True
    head = review_text.strip()[:30]
    if re.search(r"\b(false|no)\b", head, re.IGNORECASE):
        requires_human_review = False
    elif re.search(r"\b(true|yes)\b", head, re.IGNORECASE):
        requires_human_review = True

    return {
        "risk_level": risk_level,
        "possible_conditions": extract_list_field(sections.get("possible_conditions", "")),
        "recommended_tests": extract_list_field(sections.get("recommended_tests", "")),
        "treatment_considerations": extract_list_field(sections.get("treatment_considerations", "")),
        "clinical_reasoning": extract_scalar_field(sections.get("clinical_reasoning", "")),
        "warnings": extract_list_field(sections.get("warnings", "")),
        "requires_human_review": requires_human_review,
    }


def is_substantive(result: dict) -> bool:
    return bool(
        result["possible_conditions"] or result["recommended_tests"] or len(result["clinical_reasoning"]) > 15
    )


def safe_default():
    return {
        "risk_level": "Unknown", "possible_conditions": [], "recommended_tests": [],
        "treatment_considerations": [],
        "clinical_reasoning": "Automated reasoning unavailable — model output could not be parsed.",
        "warnings": ["LLM output was invalid or unparseable. Manual clinical review required."],
        "requires_human_review": True, "valid": False, "parse_method": "failed",
    }


def parse_cdss_output(raw_text: str) -> dict:
    raw_text = re.sub(r"```(?:json)?", "", raw_text).strip()

    line_stripped = re.sub(r'//[^\n]*', '', raw_text)
    try:
        match = re.search(r"\{.*\}", line_stripped, re.DOTALL)
        candidate = match.group(0) if match else None
        if not candidate and line_stripped.count('{') > 0:
            candidate = line_stripped[line_stripped.index('{'):].rstrip()
            candidate = re.sub(r',\s*$', '', candidate)
            candidate += ']' * max(candidate.count('[') - candidate.count(']'), 0)
            candidate += '}' * max(candidate.count('{') - candidate.count('}'), 0)
        if candidate:
            parsed = json.loads(candidate)
            risk_level = str(parsed.get("risk_level", "Unknown")).strip()
            if risk_level not in VALID_RISK_LEVELS:
                risk_level = "Unknown"
            result = {
                "risk_level": risk_level,
                "possible_conditions": list(parsed.get("possible_conditions", [])),
                "recommended_tests": list(parsed.get("recommended_tests", [])),
                "treatment_considerations": list(parsed.get("treatment_considerations", [])),
                "clinical_reasoning": str(parsed.get("clinical_reasoning", "")).strip(),
                "warnings": list(parsed.get("warnings", [])),
                "requires_human_review": bool(parsed.get("requires_human_review", True)),
            }
            if is_substantive(result):
                result["valid"] = True
                result["parse_method"] = "json"
                return result
    except Exception:
        pass

    comment_preserved = raw_text.replace("//", "  ")
    sections = universal_field_extract(comment_preserved)
    if sections:
        result = build_result_from_sections(sections)
        if is_substantive(result):
            result["valid"] = True
            result["parse_method"] = "fallback_universal"
            return result

    return safe_default()


# Safety overrides

def apply_safety_overrides(result: dict, clinical_input: dict) -> dict:
    esi = clinical_input.get("triage", {}).get("esi")

    if esi in (1, 2):
        result["requires_human_review"] = True

    if esi in (1, 2) and result["risk_level"] in ("Low", "Moderate"):
        result["warnings"].append(
            f"Model risk assessment ({result['risk_level']}) is lower than expected for ESI {esi}. Verify manually."
        )
        result["requires_human_review"] = True

    return result


# Full pipeline

def generate_cdss_llm(clinical_input: dict) -> dict:
    raw_output = generate_cdss_response(clinical_input)
    result = parse_cdss_output(raw_output)
    result = apply_safety_overrides(result, clinical_input)
    result["raw_output"] = raw_output
    return result
'''

with open(os.path.join(files_dir, "cdss_llm.py"), "w") as f:
    f.write(code_cdss_llm)

print("Saved exact cdss_llm.py base code.")

# 2. WRITE cdss_llm_api.py
code_cdss_llm_api = r'''"""
cdss_llm_api.py — FastAPI wrapper around cdss_llm.py (Phase 6).
"""

from fastapi import FastAPI
from pydantic import BaseModel
from typing import Optional, List, Dict, Any

from cdss_llm import generate_cdss_llm

app = FastAPI(title="CDSS LLM Reasoning Service")


class Patient(BaseModel):
    age: Optional[int] = None
    gender: Optional[str] = None


class Triage(BaseModel):
    esi: Optional[int] = None
    confidence: Optional[float] = None


class ClinicalInputRequest(BaseModel):
    patient: Patient
    vitals: Dict[str, Any] = {}
    symptoms: str = ""
    medical_history: List[str] = []
    triage: Triage


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/cdss/llm")
def cdss_llm_endpoint(request: ClinicalInputRequest):
    clinical_input = request.dict()
    return generate_cdss_llm(clinical_input)
'''

with open(os.path.join(files_dir, "cdss_llm_api.py"), "w") as f:
    f.write(code_cdss_llm_api)

print("Saved exact cdss_llm_api.py base code.")

# 3. RUN INFERENCE ON PATIENT 0
if files_dir not in sys.path:
    sys.path.insert(0, files_dir)

os.chdir(phase6_dir)
from clinical_input import get_clinical_input
import cdss_llm

patient_input = get_clinical_input(0)
llm_result = cdss_llm.generate_cdss_llm(patient_input)

print("\nPATIENT 0 OUTPUT")
print(json.dumps(llm_result, indent=2))

Saved exact cdss_llm.py base code.
Saved exact cdss_llm_api.py base code.


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


PATIENT 0 OUTPUT
{
  "risk_level": "Moderate",
  "possible_conditions": [
    "Head injury",
    "Possible concussion"
  ],
  "recommended_tests": [
    "CT scan of head",
    "Neurological assessment"
  ],
  "treatment_considerations": [
    "Monitoring for signs of increased intracranial pressure or neurological deficits"
  ],
  "clinical_reasoning": "The patient has experienced a fall with a head injury, which raises concern for potential head injury or concussion. Given the ESI level 3, the patient is at moderate risk but does not require immediate life-saving intervention. However, due to the head injury, further evaluation is necessary to rule out serious complications such as intracranial bleeding or concussion.",
  "warnings": [
    "Monitor for any changes in mental status or neurological symptoms."
  ],
  "requires_human_review": true,
  "valid": true,
  "parse_method": "json",
  "raw_output": "{\n  \"risk_level\": \"Moderate\",  \"possible_conditions\": [\"Head injury\", \"

# CDSS Integration + API + Evaluation

## Overview & Role
* **Goal:** Combine all individual sub-modules into a single unified Clinical Decision Support System (CDSS) pipeline, expose it via a REST API, and conduct end-to-end clinical and technical evaluations.

---

## Workflow & System Architecture

1. Patient Database

2. Clinical Input Extractor

3. Phase 5 Triage Model (ESI & Confidence)

4. Risk Engine & Qwen LLM Reasoning

5. CDSS Integration

6. Final CDSS Output

---

## Primary Tasks

1. **Pipeline Integration:**
   * Fetch and format patient clinical data (demographics, vitals, history, symptoms).
   * **Phase 5:** Obtain automated ESI triage score and prediction confidence.
   * Execute deterministic medical rule-based checks for risk scoring and immediate intervention rules.
   * Run quantized Qwen-2.5-3B LLM inference for nuanced clinical reasoning and diagnostic options.
   * Merge recommendations, pick the most conservative risk score, and apply safety overrides.

2. **API Deployment:**
   * Wrap the integrated pipeline using FastAPI to expose a `/cdss/evaluate` endpoint.

3. **Validation & Evaluation:**
   * Test across diverse clinical scenarios and edge cases.
   * Benchmark safety overrides, fallbacks, and human-review flags.

---

## Standardized Output Schema

```json
{
  "patient_id": 101,
  "triage": {
    "esi": 2,
    "confidence": 0.89
  },
  "clinical_decision_support": {
    "risk_level": "High",
    "possible_conditions": [
      "Acute Coronary Syndrome",
      "Unstable Angina"
    ],
    "recommended_tests": [
      "12-Lead ECG",
      "Troponin I/T",
      "Chest X-Ray"
    ],
    "treatment_considerations": [
      "Supplemental Oxygen if SpO2 < 94%",
      "Continuous Cardiac Monitoring",
      "Aspirin 325 mg PO"
    ],
    "clinical_reasoning": "Patient presents with acute chest pain and elevated vital signs under ESI Level 2. Conservative risk scoring escalates priority to High risk requiring immediate medical review.",
    "warnings": [
      "ESI level 2 requires mandatory human clinical review."
    ],
    "requires_human_review": true
  }
}

In [9]:
import os

phase6_dir = "/content/drive/MyDrive/MIMIC-IV-Ext Triage Instruction Corpus/Final_Files/Phase_6_Final"
files_dir = os.path.join(phase6_dir, "Files")
os.makedirs(files_dir, exist_ok=True)

code_cdss = r'''"""
cdss.py — Master Clinical Decision Support System Integration.
Integrates:
- Clinical Input Data
- Phase 5: Triage Prediction (ESI & Confidence)
- Rule-based Risk/Recommendation Engine
- Qwen LLM Reasoning Engine

Produces a unified, standardized CDSS response schema.
"""

import os
import json
import logging
from typing import Dict, Any

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("CDSS_Integration")

# Dynamic sub-module imports
try:
    from clinical_input import get_clinical_input
except ImportError:
    logger.warning("clinical_input module not found. Direct calls must pass clinical_input dicts.")
    get_clinical_input = None

try:
    from triage_predict import predict_triage
except ImportError:
    logger.warning("triage_predict module not found. Using fallback triage logic.")
    predict_triage = None

try:
    from cdss_rules import run_cdss_rules
except ImportError:
    logger.warning("cdss_rules module not found. Using fallback rules logic.")
    run_cdss_rules = None

try:
    from cdss_llm import generate_cdss_llm
except ImportError:
    logger.warning("cdss_llm module not found. Using fallback LLM mock logic.")
    generate_cdss_llm = None


def merge_list_field(llm_list: list, rule_list: list) -> list:
    """Combines list items from LLM and Rule Engine while removing duplicates and preserving order."""
    combined = []
    seen = set()

    for item in (llm_list or []) + (rule_list or []):
        if not item or not isinstance(item, str):
            continue
        cleaned = item.strip()
        cleaned_lower = cleaned.lower()
        if cleaned_lower not in seen:
            seen.add(cleaned_lower)
            combined.append(cleaned)

    return combined


def determine_highest_risk(llm_risk: str, rule_risk: str) -> str:
    """Selects the most conservative (highest priority) risk level between LLM and Rule Engine."""
    risk_hierarchy = {"Critical": 4, "High": 3, "Moderate": 2, "Low": 1, "Unknown": 0}

    l_score = risk_hierarchy.get(str(llm_risk).capitalize(), 0)
    r_score = risk_hierarchy.get(str(rule_risk).capitalize(), 0)

    highest_score = max(l_score, r_score)
    for risk_name, score in risk_hierarchy.items():
        if score == highest_score and score > 0:
            return risk_name

    return "Unknown"


def run_cdss_pipeline(patient_id: int, clinical_input: Dict[str, Any] = None) -> Dict[str, Any]:
    """
    Executes the complete end-to-end CDSS integration pipeline.

    Args:
        patient_id (int): Identifier for the patient.
        clinical_input (dict, optional): Clinical payload. If None, retrieved via Person A loader.

    Returns:
        dict: Standardized CDSS output.
    """
    logger.info(f"Executing CDSS integration pipeline for Patient ID: {patient_id}")

    # 1. Fetch Clinical Input (Person A)
    if clinical_input is None:
        if get_clinical_input is not None:
            try:
                clinical_input = get_clinical_input(patient_id)
            except Exception as e:
                logger.error(f"Failed to fetch clinical input for patient {patient_id}: {e}")
                clinical_input = {}
        else:
            clinical_input = {}

    if "patient" not in clinical_input:
        clinical_input["patient"] = {"patient_id": patient_id}
    else:
        clinical_input["patient"]["patient_id"] = patient_id

    # 2. Extract or Predict Triage (Phase 5)
    triage_info = clinical_input.get("triage", {})
    esi = triage_info.get("esi")
    confidence = triage_info.get("confidence", 0.0)

    if (esi is None or confidence == 0.0) and predict_triage is not None:
        try:
            triage_res = predict_triage(clinical_input)
            esi = triage_res.get("esi", esi)
            confidence = triage_res.get("confidence", confidence)
        except Exception as e:
            logger.error(f"Phase 5 Triage prediction error: {e}")

    if esi is None:
        esi = 3
        confidence = 0.50
        logger.warning(f"Defaulting ESI to 3 (Confidence: 0.50) for patient {patient_id}")

    clinical_input["triage"] = {"esi": esi, "confidence": confidence}

    # 3. Execute Rule Engine (Person B)
    rule_output = {}
    if run_cdss_rules is not None:
        try:
            rule_output = run_cdss_rules(clinical_input)
        except Exception as e:
            logger.error(f"Person B Rule Engine execution error: {e}")

    # 4. Execute Qwen LLM Reasoning (Person C)
    llm_output = {}
    if generate_cdss_llm is not None:
        try:
            llm_output = generate_cdss_llm(clinical_input)
        except Exception as e:
            logger.error(f"Person C LLM Reasoning Engine execution error: {e}")
            llm_output = {
                "risk_level": "Unknown",
                "possible_conditions": [],
                "recommended_tests": [],
                "treatment_considerations": [],
                "clinical_reasoning": f"LLM execution failed: {str(e)}",
                "warnings": ["LLM service unavailable. Rule-based assessment applied."],
                "requires_human_review": True
            }

    # 5. Merge Outputs & Determine Conservative Risk Level
    final_risk = determine_highest_risk(
        llm_output.get("risk_level", "Unknown"),
        rule_output.get("risk_level", "Unknown")
    )

    combined_conditions = merge_list_field(
        llm_output.get("possible_conditions", []),
        rule_output.get("possible_conditions", [])
    )

    combined_tests = merge_list_field(
        llm_output.get("recommended_tests", []),
        rule_output.get("recommended_tests", [])
    )

    combined_treatments = merge_list_field(
        llm_output.get("treatment_considerations", []),
        rule_output.get("treatment_considerations", [])
    )

    combined_warnings = merge_list_field(
        llm_output.get("warnings", []),
        rule_output.get("warnings", [])
    )

    # Human Review Safety Rules: ESI 1 or 2, Critical/High Risk, or explicit module flag
    requires_review = (
        bool(llm_output.get("requires_human_review", False)) or
        bool(rule_output.get("requires_human_review", False)) or
        (esi in [1, 2]) or
        (final_risk in ["High", "Critical"])
    )

    reasoning = llm_output.get("clinical_reasoning", "")
    if not reasoning and rule_output.get("rule_summary"):
        reasoning = rule_output.get("rule_summary")

    # Build standardized output schema
    final_response = {
        "patient_id": patient_id,
        "triage": {
            "esi": int(esi),
            "confidence": float(round(confidence, 2))
        },
        "clinical_decision_support": {
            "risk_level": final_risk,
            "possible_conditions": combined_conditions,
            "recommended_tests": combined_tests,
            "treatment_considerations": combined_treatments,
            "clinical_reasoning": reasoning,
            "warnings": combined_warnings,
            "requires_human_review": requires_review
        }
    }

    return final_response
'''

with open(os.path.join(files_dir, "cdss.py"), "w") as f:
    f.write(code_cdss)

print("cdss.py saved successfully.")

cdss.py saved successfully.


In [10]:
import os

phase6_dir = "/content/drive/MyDrive/MIMIC-IV-Ext Triage Instruction Corpus/Final_Files/Phase_6_Final"
files_dir = os.path.join(phase6_dir, "Files")

code_cdss_api = r'''"""
cdss_api.py — FastAPI REST Service for Integrated CDSS.
Exposes /cdss/evaluate endpoint combining all.
"""

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional, List, Dict, Any

from cdss import run_cdss_pipeline

app = FastAPI(
    title="Integrated Clinical Decision Support System API",
    description="Person D Integration API combining Clinical Input, Triage, Rules, and LLM Reasoning.",
    version="1.0.0"
)


class PatientSchema(BaseModel):
    patient_id: Optional[int] = 101
    age: Optional[int] = None
    gender: Optional[str] = None


class TriageSchema(BaseModel):
    esi: Optional[int] = None
    confidence: Optional[float] = None


class CDSSRequestSchema(BaseModel):
    patient_id: Optional[int] = 101
    patient: Optional[PatientSchema] = Field(default_factory=PatientSchema)
    vitals: Dict[str, Any] = {}
    symptoms: str = ""
    medical_history: List[str] = []
    triage: Optional[TriageSchema] = Field(default_factory=TriageSchema)


@app.get("/health")
def health_check():
    return {"status": "ok", "service": "Integrated CDSS Service"}


@app.post("/cdss/evaluate")
def evaluate_cdss_endpoint(request: CDSSRequestSchema):
    try:
        payload = request.dict()
        patient_id = payload.get("patient_id") or payload.get("patient", {}).get("patient_id", 101)

        result = run_cdss_pipeline(patient_id=patient_id, clinical_input=payload)
        return result
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"CDSS Engine Error: {str(e)}")
'''

with open(os.path.join(files_dir, "cdss_api.py"), "w") as f:
    f.write(code_cdss_api)

print("cdss_api.py saved successfully.")

cdss_api.py saved successfully.
